# AI Data Leak Detector

An AI-assisted security tool that scans text for potential PII, PHI, credentials, and secrets.

This version includes local redaction to reduce sensitive-data exposure before API transmission, input validation, and API error handling.

## 1. Setup

Install and import the libraries required for the scanner.

In [5]:
!pip install openai -q

In [1]:
from openai import OpenAI
from getpass import getpass
import os
import re
import json
import textwrap

## 2. API Authentication

Securely load the OpenAI API key from a masked input and initialize the API client.

In [3]:
os.environ["OPENAI_API_KEY"] = getpass("API Key: ")

API Key: ··········


In [4]:
client = OpenAI()

## 3. User Input & Validation

Collect text from the user and check that data was entered before processing.

In [5]:
print("Paste internal data to scan.")
print("Type END on a new line when finished.\n")

lines = []

while True:
    line = input()

    if line.strip().upper() == "END":
        break

    lines.append(line)

data = "\n".join(lines).strip()

Paste internal data to scan.
Type END on a new line when finished.

Employee: Test User
SSN: 555 12 3456
Email: test.user@example.com
Phone: 5558675309
API Key: sk-FAKE1234567890TEST
Password: FakePassword123!
END


In [6]:
if not data:
    print("Error: No data was entered. Please enter text to scan.")
else:
    print("Data received. Ready to scan.")

Data received. Ready to scan.


## 4. Local Sensitive Data Redaction

Detect and redact obvious sensitive-data patterns locally before sending text to the AI service.

In [7]:
ssn_pattern = r"\b\d{3}[-\s]\d{2}[-\s]\d{4}\b"

email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

phone_pattern = r"\b(?:\+?1[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b"

api_key_pattern = r"\bsk-[A-Za-z0-9_-]{10,}\b"

password_pattern = r"(?i)\b(password|passwd|pwd|secret|access[_\s-]?token)\s*[:=]\s*\S+"

In [8]:
redacted_data = re.sub(ssn_pattern, "[REDACTED-SSN]", data)
redacted_data = re.sub(email_pattern, "[REDACTED-EMAIL]", redacted_data)
redacted_data = re.sub(phone_pattern, "[REDACTED-PHONE]", redacted_data)
redacted_data = re.sub(api_key_pattern, "[REDACTED-API-KEY]", redacted_data)
redacted_data = re.sub(password_pattern, "[REDACTED-PASSWORD]", redacted_data)

In [9]:
print(redacted_data)

Employee: Test User
SSN: [REDACTED-SSN]
Email: [REDACTED-EMAIL]
Phone: [REDACTED-PHONE]
API Key: [REDACTED-API-KEY]
[REDACTED-PASSWORD]


## 5. AI Security Analysis Prompt

Build a structured security-audit prompt using the locally redacted data.

In [10]:
prompt = f"""
You are a data security auditor.

Analyze the following internal data for sensitive information.

Check for PII, PHI, secrets, and credentials.

Return ONLY valid JSON. Do not include markdown, code fences, or text outside the JSON.

Use exactly this structure:

{{
  "score": 0,
  "findings": [
    {{
      "data_type": "type of sensitive data",
      "evidence": "redacted evidence only",
      "risk": "why this finding is risky"
    }}
  ],
  "remediation": [
    "recommended action"
  ]
}}

The score must be an integer from 1 to 100, where higher numbers indicate greater risk.

When a finding may be subject to a specific privacy, security, or data protection regulation, identify the relevant regulation in the risk explanation.

Do not invent a regulation if one does not clearly apply.

Do not reproduce raw passwords, credentials, secrets, PII, or PHI in your response.

Data:

{redacted_data}
"""

## 6. Run AI Analysis

Send the sanitized prompt to the AI model and return the security analysis. API errors are handled gracefully to prevent the notebook from crashing.

In [11]:
try:
    response = client.responses.create(
        model="gpt-5-mini",
        input=prompt
    )

    raw_output = response.output_text.strip()

    # Remove markdown code fences if the model adds them
    if raw_output.startswith("```"):
        raw_output = raw_output.strip("`")
        if raw_output.lower().startswith("json"):
            raw_output = raw_output[4:].strip()

    analysis = json.loads(raw_output)

    print("=" * 70)
    print("SECURITY ANALYSIS")
    print("=" * 70)

    print(f"\nRisk Score: {analysis['score']}/100")

    print("\nFINDINGS")
    print("-" * 70)

    for number, finding in enumerate(analysis["findings"], start=1):
        print(f"\nFinding {number}")
        print(f"Data Type: {finding['data_type']}")
        print(f"Evidence:  {finding['evidence']}")

        risk_text = textwrap.fill(
            finding["risk"],
            width=90,
            initial_indent="Risk:      ",
            subsequent_indent="           "
        )
        print(risk_text)

    print("\nREMEDIATION")
    print("-" * 70)

    for number, action in enumerate(analysis["remediation"], start=1):
        formatted_action = textwrap.fill(
            action,
            width=90,
            initial_indent=f"{number}. ",
            subsequent_indent="   "
        )
        print(formatted_action)

except json.JSONDecodeError:
    print("Error: The AI response was not valid JSON.")

except Exception as error:
    print("Error: The AI analysis could not be completed.")
    print(error)

SECURITY ANALYSIS

Risk Score: 85/100

FINDINGS
----------------------------------------------------------------------

Finding 1
Data Type: Personally Identifiable Information (Name)
Evidence:  [REDACTED-EMPLOYEE-NAME]
Risk:      Contains an employee's full name that can be combined with other data for
           profiling, targeted phishing, or social engineering attacks. If the individual
           is within the scope of data-protection laws (e.g., GDPR or CCPA), processing
           must follow those rules.

Finding 2
Data Type: Sensitive PII (Social Security Number)
Evidence:  [REDACTED-SSN]
Risk:      Social Security Numbers are highly sensitive and can be used for identity theft
           and financial fraud. Unauthorized exposure or misuse can trigger breach-
           notification obligations and civil liability under various U.S. federal and
           state laws; additional protections may apply depending on the individual's
           jurisdiction.

Finding 3
Data Type: